# 🥈 Silver Layer - Dimension Tables
## AWS S3 External Storage via Unity Catalog

**Purpose**: Build cleaned, conformed dimension tables from Bronze layer with SCD Type 2

**Source**: `workspace.`workspace-adventureworks-bronze`` (S3)
**Target**: `workspace.`workspace-adventureworks-silver`` (S3)

**Dimension Tables to Create**:
1. **dim_customer** - Customer master data with attributes
2. **dim_product** - Product catalog with category hierarchy
3. **dim_date** - Date dimension for time intelligence
4. **dim_location** - Geographic locations (addresses, regions)
5. **dim_employee** - Employee information

**Features**:
- Data quality checks and cleansing
- SCD Type 2 (slowly changing dimensions) with effective dates
- Business key deduplication
- Surrogate key generation
- Conformance to naming standards

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime, date
import time

# Configuration
BRONZE_SCHEMA = "workspace.`workspace-adventureworks-bronze`"
SILVER_SCHEMA = "workspace.`workspace-adventureworks-silver`"

print("=" * 80)
print("🥈 SILVER LAYER - DIMENSION TABLES")
print("=" * 80)
print(f"Start Time: {datetime.now()}")
print(f"Source: {BRONZE_SCHEMA} (AWS S3)")
print(f"Target: {SILVER_SCHEMA} (AWS S3)")
print("=" * 80)
print()

# Track transformation results
transformation_results = []

In [0]:
def create_scd_type2_dimension(df, business_key_cols, table_name):
    """
    Create SCD Type 2 dimension table with surrogate keys and effective dates
    
    Args:
        df: Input DataFrame from Bronze
        business_key_cols: List of columns that form the business key
        table_name: Name of the dimension table
    
    Returns:
        DataFrame with SCD Type 2 columns added
    """
    # Add SCD Type 2 columns
    df_scd = (df
        .withColumn("effective_start_date", current_timestamp())
        .withColumn("effective_end_date", lit(None).cast("timestamp"))
        .withColumn("is_current", lit(True))
        .withColumn("row_hash", md5(concat_ws("|", *[col(c) for c in df.columns])))
    )
    
    # Generate surrogate key using row_number
    window_spec = Window.orderBy("ingestion_timestamp")
    df_scd = df_scd.withColumn(
        f"{table_name}_sk",
        row_number().over(window_spec)
    )
    
    # Reorder columns: surrogate key first, then business key, then attributes, then SCD columns
    scd_cols = ["effective_start_date", "effective_end_date", "is_current", "row_hash", "ingestion_timestamp", "source_system"]
    other_cols = [c for c in df_scd.columns if c not in business_key_cols + scd_cols + [f"{table_name}_sk"]]
    
    final_cols = [f"{table_name}_sk"] + business_key_cols + other_cols + scd_cols
    
    return df_scd.select([c for c in final_cols if c in df_scd.columns])

def clean_and_deduplicate(df, business_key_cols, description="table"):
    """
    Clean data and remove duplicates based on business key
    """
    initial_count = df.count()
    
    # Remove nulls in business key columns
    for col_name in business_key_cols:
        df = df.filter(col(col_name).isNotNull())
    
    # Deduplicate by business key (keep most recent)
    window_spec = Window.partitionBy(business_key_cols).orderBy(col("ingestion_timestamp").desc())
    df_clean = (df
        .withColumn("row_num", row_number().over(window_spec))
        .filter(col("row_num") == 1)
        .drop("row_num")
    )
    
    final_count = df_clean.count()
    removed = initial_count - final_count
    
    print(f"   Data Quality: {description}")
    print(f"     Initial: {initial_count:,} rows")
    print(f"     After cleaning: {final_count:,} rows")
    print(f"     Removed: {removed:,} rows ({removed/initial_count*100:.1f}%)")
    
    return df_clean

In [0]:
# Build Customer Dimension
print("\n" + "=" * 80)
print("👥 BUILDING dim_customer")
print("=" * 80)

start_time = time.time()

try:
    # Read from Bronze
    customer = spark.table(f"{BRONZE_SCHEMA}.customer")
    
    # Select and rename columns
    df_customer = customer.select(
        col("CustomerID").alias("customer_id"),
        col("Title").alias("title"),
        col("FirstName").alias("first_name"),
        col("MiddleName").alias("middle_name"),
        col("LastName").alias("last_name"),
        col("Suffix").alias("suffix"),
        col("CompanyName").alias("company_name"),
        col("SalesPerson").alias("sales_person"),
        col("EmailAddress").alias("email_address"),
        col("Phone").alias("phone"),
        col("PasswordHash"),
        col("PasswordSalt"),
        col("ModifiedDate").alias("modified_date"),
        col("ingestion_timestamp"),
        col("source_system")
    )
    
    # Clean and deduplicate
    df_customer = clean_and_deduplicate(df_customer, ["customer_id"], "Customer")
    
    # Apply SCD Type 2
    df_customer_scd = create_scd_type2_dimension(df_customer, ["customer_id"], "dim_customer")
    
    # Write to Silver
    (df_customer_scd
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_SCHEMA}.dim_customer")
    )
    
    duration = time.time() - start_time
    row_count = df_customer_scd.count()
    
    transformation_results.append({
        "table": "dim_customer",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ dim_customer created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed to create dim_customer: {str(e)}")
    transformation_results.append({"table": "dim_customer", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Build Product Dimension with Category Hierarchy
print("\n" + "=" * 80)
print("📦 BUILDING dim_product")
print("=" * 80)

start_time = time.time()

try:
    # Read from Bronze
    product = spark.table(f"{BRONZE_SCHEMA}.product")
    product_category = spark.table(f"{BRONZE_SCHEMA}.productcategory")
    product_model = spark.table(f"{BRONZE_SCHEMA}.productmodel").select(
        col("ProductModelID"), col("Name").alias("model_name")
    )
    
    # Join product with category and model
    df_product = (product
        .join(product_category, "ProductCategoryID", "left")
        .join(product_model, "ProductModelID", "left")
        .select(
            col("ProductID").alias("product_id"),
            col("Name").alias("product_name"),
            col("ProductNumber").alias("product_number"),
            col("Color").alias("color"),
            col("StandardCost").alias("standard_cost"),
            col("ListPrice").alias("list_price"),
            col("Size").alias("size"),
            col("Weight").alias("weight"),
            col("ProductCategoryID").alias("product_category_id"),
            col("product_category.Name").alias("category_name"),
            col("ProductModelID").alias("product_model_id"),
            col("model_name"),
            col("SellStartDate").alias("sell_start_date"),
            col("SellEndDate").alias("sell_end_date"),
            col("DiscontinuedDate").alias("discontinued_date"),
            col("product.ModifiedDate").alias("modified_date"),
            col("product.ingestion_timestamp"),
            col("product.source_system")
        )
    )
    
    # Clean and deduplicate
    df_product = clean_and_deduplicate(df_product, ["product_id"], "Product")
    
    # Apply SCD Type 2
    df_product_scd = create_scd_type2_dimension(df_product, ["product_id"], "dim_product")
    
    # Write to Silver
    (df_product_scd
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_SCHEMA}.dim_product")
    )
    
    duration = time.time() - start_time
    row_count = df_product_scd.count()
    
    transformation_results.append({
        "table": "dim_product",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ dim_product created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed to create dim_product: {str(e)}")
    transformation_results.append({"table": "dim_product", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Build Date Dimension
print("\n" + "=" * 80)
print("📅 BUILDING dim_date")
print("=" * 80)

start_time = time.time()

try:
    # Generate date range (2000-2030)
    from datetime import timedelta
    
    start_date = date(2000, 1, 1)
    end_date = date(2030, 12, 31)
    
    # Create date range
    date_list = []
    current_date = start_date
    while current_date <= end_date:
        date_list.append((current_date,))
        current_date += timedelta(days=1)
    
    df_dates = spark.createDataFrame(date_list, ["date_value"])
    
    # Add date attributes
    df_date = (df_dates
        .withColumn("date_sk", row_number().over(Window.orderBy("date_value")))
        .withColumn("year", year("date_value"))
        .withColumn("quarter", quarter("date_value"))
        .withColumn("month", month("date_value"))
        .withColumn("month_name", date_format("date_value", "MMMM"))
        .withColumn("day_of_month", dayofmonth("date_value"))
        .withColumn("day_of_week", dayofweek("date_value"))
        .withColumn("day_name", date_format("date_value", "EEEE"))
        .withColumn("week_of_year", weekofyear("date_value"))
        .withColumn("is_weekend", when(dayofweek("date_value").isin([1, 7]), True).otherwise(False))
        .withColumn("fiscal_year", when(month("date_value") >= 7, year("date_value") + 1).otherwise(year("date_value")))
        .withColumn("fiscal_quarter", 
            when(month("date_value").between(7, 9), 1)
            .when(month("date_value").between(10, 12), 2)
            .when(month("date_value").between(1, 3), 3)
            .otherwise(4)
        )
        .select(
            "date_sk",
            "date_value",
            "year",
            "quarter",
            "month",
            "month_name",
            "day_of_month",
            "day_of_week",
            "day_name",
            "week_of_year",
            "is_weekend",
            "fiscal_year",
            "fiscal_quarter"
        )
    )
    
    # Write to Silver
    (df_date
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_SCHEMA}.dim_date")
    )
    
    duration = time.time() - start_time
    row_count = df_date.count()
    
    transformation_results.append({
        "table": "dim_date",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ dim_date created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed to create dim_date: {str(e)}")
    transformation_results.append({"table": "dim_date", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Build Location Dimension
print("\n" + "=" * 80)
print("🌎 BUILDING dim_location")
print("=" * 80)

start_time = time.time()

try:
    # Read from Bronze
    address = spark.table(f"{BRONZE_SCHEMA}.address")
    
    # Build location dimension
    df_location = address.select(
        col("AddressID").alias("address_id"),
        col("AddressLine1").alias("address_line1"),
        col("AddressLine2").alias("address_line2"),
        col("City").alias("city"),
        col("StateProvince").alias("state_province"),
        col("CountryRegion").alias("country_region"),
        col("PostalCode").alias("postal_code"),
        col("ModifiedDate").alias("modified_date"),
        col("ingestion_timestamp"),
        col("source_system")
    )
    
    # Clean and deduplicate
    df_location = clean_and_deduplicate(df_location, ["address_id"], "Location")
    
    # Apply SCD Type 2
    df_location_scd = create_scd_type2_dimension(df_location, ["address_id"], "dim_location")
    
    # Write to Silver
    (df_location_scd
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_SCHEMA}.dim_location")
    )
    
    duration = time.time() - start_time
    row_count = df_location_scd.count()
    
    transformation_results.append({
        "table": "dim_location",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ dim_location created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed to create dim_location: {str(e)}")
    transformation_results.append({"table": "dim_location", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Build Employee Dimension
print("\n" + "=" * 80)
print("👤 BUILDING dim_employee")
print("=" * 80)

start_time = time.time()

try:
    # Read from Bronze
    employee = spark.table(f"{BRONZE_SCHEMA}.employee")
    
    # Build employee dimension
    df_employee = employee.select(
        col("BusinessEntityID").alias("employee_id"),
        col("NationalIDNumber").alias("national_id"),
        col("LoginID").alias("login_id"),
        col("JobTitle").alias("job_title"),
        col("BirthDate").alias("birth_date"),
        col("MaritalStatus").alias("marital_status"),
        col("Gender").alias("gender"),
        col("HireDate").alias("hire_date"),
        col("SalariedFlag").alias("is_salaried"),
        col("VacationHours").alias("vacation_hours"),
        col("SickLeaveHours").alias("sick_leave_hours"),
        col("CurrentFlag").alias("is_current_employee"),
        col("ModifiedDate").alias("modified_date"),
        col("ingestion_timestamp"),
        col("source_system")
    )
    
    # Clean and deduplicate
    df_employee = clean_and_deduplicate(df_employee, ["employee_id"], "Employee")
    
    # Apply SCD Type 2
    df_employee_scd = create_scd_type2_dimension(df_employee, ["employee_id"], "dim_employee")
    
    # Write to Silver
    (df_employee_scd
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_SCHEMA}.dim_employee")
    )
    
    duration = time.time() - start_time
    row_count = df_employee_scd.count()
    
    transformation_results.append({
        "table": "dim_employee",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ dim_employee created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed to create dim_employee: {str(e)}")
    transformation_results.append({"table": "dim_employee", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Summary
from pyspark.sql import Row

print("\n" + "=" * 80)
print("📊 DIMENSION TRANSFORMATION SUMMARY")
print("=" * 80)

success_count = sum(1 for r in transformation_results if r["status"] == "success")
failed_count = sum(1 for r in transformation_results if r["status"] == "failed")
total_rows = sum(r["row_count"] for r in transformation_results)

print(f"\nDimensions Processed: {len(transformation_results)}")
print(f"  ✅ Success: {success_count}")
print(f"  ❌ Failed: {failed_count}")
print(f"Total Rows Created: {total_rows:,}")

summary_df = spark.createDataFrame([Row(**r) for r in transformation_results])
print("\nDetailed Results:")
display(summary_df.orderBy("status", "table"))

print("\n" + "=" * 80)
if failed_count == 0:
    print("✅ SILVER DIMENSIONS COMPLETED SUCCESSFULLY")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit("SUCCESS")
else:
    print("⚠️ SILVER DIMENSIONS COMPLETED WITH ERRORS")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit(f"PARTIAL_SUCCESS: {failed_count} dimensions failed")